In [1]:
import os
import json
import shutil
from datetime import datetime
from pathlib import Path


DAY06_PATH = Path(
    r"D:\Anaconda\Project\AI-Coding-Agent\agent-learning\day06"
)

WORK_PATH = Path(
    r"D:\Anaconda\Project\AI-Coding-Agent\agent-learning\work"
)

GENERATED_PATH = (
    DAY06_PATH
    /
    "generated"
)


os.chdir(
    DAY06_PATH
)


from memory.state import AgentState
from workflow.graph import AgentWorkflow
from workflow.review_router import review_router
from agents.unity_compiler import unity_compile_agent


workflow = AgentWorkflow()

code_checker = workflow.code_checker
reviewer = workflow.reviewer
repair = workflow.repair


required_fields = {
    "compile_history",
    "review_history",
    "review_retry_count",
    "repair_history",
    "repair_count"
}


assert required_fields.issubset(
    AgentState.__annotations__
)


print(
    "PASS: Day06-4 workflow 初始化成功"
)

PASS: Day06-4 workflow 初始化成功


In [2]:
routing_results = {
    "compile_failure_to_repair":
        review_router({
            "review": {
                "score": 95,
                "pass": False,
                "remaining_issues": [
                    {
                        "file": "Test.cs",
                        "problem": "CS1002"
                    }
                ]
            },
            "code_check_result": {
                "success": True
            },
            "compile_result": {
                "success": False,
                "system_error": False
            },
            "repair_count": 0,
            "review_retry_count": 0
        }) == "repair",

    "strict_pass_to_finish":
        review_router({
            "review": {
                "score": 100,
                "pass": True,
                "remaining_issues": []
            },
            "code_check_result": {
                "success": True
            },
            "compile_result": {
                "success": True,
                "system_error": False
            },
            "repair_count": 1,
            "review_retry_count": 0
        }) == "finish_task",

    "system_error_stops":
        workflow.unity_compiler_router({
            "compile_result": {
                "success": False,
                "system_error": True
            }
        }) == "finish_task"
}


assert all(
    routing_results.values()
), routing_results


routing_results

[Review Router]评分:95,问题数量:1,修复次数:0,Review重试:0
[Review Router]Unity编译失败，进入代码修复
[Review Router]评分:100,问题数量:0,修复次数:1,Review重试:0
[Review Router]审核通过
[Unity Compiler Router]系统错误，终止修复循环


{'compile_failure_to_repair': True,
 'strict_pass_to_finish': True,
 'system_error_stops': True}

In [3]:
backup_name = (
    "day06_generated_backup_"
    +
    datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
)


BACKUP_PATH = (
    WORK_PATH
    /
    backup_name
)


assert GENERATED_PATH.exists()


shutil.copytree(
    GENERATED_PATH,
    BACKUP_PATH
)


PROBE_FILE = (
    GENERATED_PATH
    /
    "Day06ClosureProbe.cs"
)


PROBE_FILE.write_text(
    """
using UnityEngine;

public class Day06ClosureProbe : MonoBehaviour
{
    private void Start()
    {
        int brokenValue = ;
        Debug.Log(brokenValue);
    }
}
""".strip(),
    encoding="utf-8"
)


assert PROBE_FILE.exists()


print(
    "Backup:",
    BACKUP_PATH
)


print(
    "Injected:",
    PROBE_FILE
)

Backup: D:\Anaconda\Project\AI-Coding-Agent\agent-learning\work\day06_generated_backup_20260805_174658
Injected: D:\Anaconda\Project\AI-Coding-Agent\agent-learning\day06\generated\Day06ClosureProbe.cs


In [4]:
closed_loop_state = {
    "query":
        "修复 Day06ClosureProbe.cs 中的 Unity C# 编译错误",

    "tasks": [],

    "current_agent": "",

    "agent_history": [],

    "requirements": [],

    "context": [],

    "architecture":
        "Unity compile and repair closure test",

    "architecture_validation": {},

    "files": [],

    "code": [],

    "code_check_result": {},

    "compile_result": {},

    "compile_history": [],

    "review": {},

    "review_history": [],

    "review_retry_count": 0,

    "root_causes": [],

    "repair_count": 0,

    "repair_status": "",

    "repair_result": {},

    "repair_history": [],

    "tools": [],

    "tokens": 0
}


final_route = None


print(
    "PASS: 闭环状态初始化完成"
)

PASS: 闭环状态初始化完成


In [5]:
max_compile_rounds = 4
max_review_attempts = 2


try:

    for compile_round in range(
        1,
        max_compile_rounds + 1
    ):

        print(
            "\n"
            +
            "=" * 60
        )

        print(
            f"Compile round {compile_round}"
        )

        print(
            "=" * 60
        )


        # 1. 读取当前修复后的文件并执行静态检查
        closed_loop_state.update(
            code_checker.run(
                closed_loop_state
            )
        )


        # 2. 真实 Unity 编译
        closed_loop_state = unity_compile_agent(
            closed_loop_state
        )


        compile_result = (
            closed_loop_state.get(
                "compile_result",
                {}
            )
        )


        print(
            "Compile success:",
            compile_result.get(
                "success",
                False
            )
        )


        print(
            "System error:",
            compile_result.get(
                "system_error",
                False
            )
        )


        print(
            "Compiler errors:",
            json.dumps(
                compile_result.get(
                    "errors",
                    []
                ),
                ensure_ascii=False,
                indent=2
            )
        )


        # 环境错误不能交给 Repair
        if compile_result.get(
            "system_error",
            False
        ):

            final_route = "system_error"
            break


        # 3. Reviewer，最多允许一次格式重试
        next_route = "reviewer"


        for review_attempt in range(
            1,
            max_review_attempts + 1
        ):

            print(
                f"\nReviewer attempt {review_attempt}"
            )


            review_update = reviewer.run(
                closed_loop_state
            )


            closed_loop_state.update(
                review_update
            )


            next_route = review_router(
                closed_loop_state
            )


            print(
                "Reviewer score:",
                closed_loop_state
                .get("review", {})
                .get("score")
            )


            print(
                "Reviewer pass:",
                closed_loop_state
                .get("review", {})
                .get("pass")
            )


            print(
                "Remaining issues:",
                closed_loop_state
                .get("review", {})
                .get(
                    "remaining_issues",
                    []
                )
            )


            print(
                "Next route:",
                next_route
            )


            if next_route != "reviewer":
                break


        final_route = next_route


        # 已严格通过
        if final_route == "finish_task":

            break


        # 架构路由不属于本次 Repair 闭环
        if final_route == "architecture":

            print(
                "STOP: Reviewer 返回架构级问题"
            )

            break


        # Reviewer 两次仍然无效
        if final_route == "reviewer":

            print(
                "STOP: Reviewer 重试后仍未产生有效结果"
            )

            break


        # 4. 执行真实 RepairAgent
        if final_route == "repair":

            repair_update = repair.run(
                closed_loop_state
            )


            closed_loop_state.update(
                repair_update
            )


            print(
                "Repair count:",
                closed_loop_state.get(
                    "repair_count",
                    0
                )
            )


            print(
                "Latest repair:",
                json.dumps(
                    closed_loop_state
                    .get(
                        "repair_history",
                        []
                    )[-1],
                    ensure_ascii=False,
                    indent=2
                )
            )


            continue


        print(
            "STOP: 未知路由",
            final_route
        )

        break


finally:

    # 无论测试成功还是失败，都恢复 generated
    if GENERATED_PATH.exists():

        shutil.rmtree(
            GENERATED_PATH
        )


    shutil.copytree(
        BACKUP_PATH,
        GENERATED_PATH
    )


    print(
        "\nGenerated files restored from:",
        BACKUP_PATH
    )


closed_loop_state[
    "current_agent"
] = final_route


print(
    "\nRepair loop execution completed."
)


Compile round 1
[Code Checker Agent]开始执行
[Code Checker Agent]代码检查通过
[Unity Compiler Agent]开始执行
[Unity Compiler]发现错误:1
Compile success: False
System error: False
Compiler errors: [
  {
    "file": "Day06ClosureProbe.cs",
    "line": 7,
    "code": "CS1525",
    "message": "Invalid expression term ';'"
  }
]

Reviewer attempt 1
[Reviewer Agent]开始执行
[DeepSeek]请求模型，第1次
[DeepSeek]调用成功
[Reviewer Raw Output]
{
    "score": 60,
    "pass": false,
    "root_causes": [
        {
            "id": 1,
            "type": "missing_reference",
            "symbol": "InventoryItemData",
            "source_file": "InventoryView.cs",
            "target_file": "InventoryView.cs",
            "affected_methods": [
                "RefreshView",
                "UpdateWeightDisplay",
                "ShowItemDetails"
            ],
            "error_code": "CS0246",
            "description": "InventoryView.cs 中引用了 InventoryItemData 类型，但该类型定义在 InventoryView.cs 文件内部的 Game.Inventory 命名空间中，而 InventoryVie

In [6]:
review_result = (
    closed_loop_state.get(
        "review",
        {}
    )
)


compile_history = (
    closed_loop_state.get(
        "compile_history",
        []
    )
)


repair_history = (
    closed_loop_state.get(
        "repair_history",
        []
    )
)


summary = {
    "compile_history":
        compile_history,

    "repair_count":
        closed_loop_state.get(
            "repair_count",
            0
        ),

    "repair_history":
        repair_history,

    "code_check_success":
        closed_loop_state
        .get("code_check_result", {})
        .get("success", False),

    "compile_success":
        closed_loop_state
        .get("compile_result", {})
        .get("success", False),

    "system_error":
        closed_loop_state
        .get("compile_result", {})
        .get("system_error", False),

    "review_score":
        review_result.get(
            "score",
            0
        ),

    "review_pass":
        review_result.get(
            "pass",
            False
        ),

    "remaining_issues":
        review_result.get(
            "remaining_issues",
            []
        ),

    "review_retry_count":
        closed_loop_state.get(
            "review_retry_count",
            0
        ),

    "root_causes":
        closed_loop_state.get(
            "root_causes",
            []
        ),

    "current_agent":
        final_route
}


print(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2
    )
)

{
  "compile_history": [
    {
      "round": 1,
      "success": false,
      "error_count": 1,
      "system_error": false
    },
    {
      "round": 2,
      "success": true,
      "error_count": 0,
      "system_error": false
    }
  ],
  "repair_count": 1,
  "repair_history": [
    {
      "round": 1,
      "actions": [
        {
          "type": "llm",
          "success": true,
          "files": [
            "Day06ClosureProbe.cs"
          ],
          "root": {
            "id": 1,
            "type": "compile_error",
            "symbol": "",
            "source_file": "Day06ClosureProbe.cs",
            "target_file": "Day06ClosureProbe.cs",
            "affected_methods": [],
            "error_code": "CS1525",
            "fix_action": {
              "operation": "repair_compile_errors",
              "target": "Day06ClosureProbe.cs",
              "details": "CS1525:Invalid expression term ';'"
            },
            "fix_strategy": "CS1525:Invalid expression ter

In [7]:
compile_failed_first = (
    len(compile_history) >= 2
    and
    compile_history[0].get(
        "success"
    )
    is False
)


compile_passed_after_repair = (
    compile_history[-1].get(
        "success"
    )
    is True
)


repair_really_executed = (
    closed_loop_state.get(
        "repair_count",
        0
    )
    >= 1
    and
    len(
        repair_history
    )
    >= 1
    and
    any(
        action.get(
            "success",
            False
        )
        for repair_round
        in repair_history
        for action
        in repair_round.get(
            "actions",
            []
        )
    )
)


review_really_passed = (
    review_result.get(
        "pass",
        False
    )
    and
    review_result.get(
        "score",
        0
    ) >= 90
    and
    review_result.get(
        "remaining_issues",
        []
    ) == []
)


actual_pass = (
    compile_failed_first
    and
    compile_passed_after_repair
    and
    repair_really_executed
    and
    closed_loop_state
    .get("code_check_result", {})
    .get("success", False)
    and
    not closed_loop_state
    .get("compile_result", {})
    .get("system_error", False)
    and
    review_really_passed
    and
    final_route == "finish_task"
)


assert compile_failed_first, (
    "第一轮没有产生真实编译失败"
)


assert repair_really_executed, (
    "RepairAgent 没有真实执行成功"
)


assert compile_passed_after_repair, (
    "Repair 后 Unity 编译仍未通过"
)


assert review_really_passed, (
    "最终 Reviewer 没有严格通过"
)


assert final_route == "finish_task", (
    f"最终路由不是 finish_task: {final_route}"
)


assert actual_pass is True


print(
    "PASS: Day06-4 真实完整修复闭环测试通过"
)

PASS: Day06-4 真实完整修复闭环测试通过


In [8]:
cleanup_state = {
    "compile_history": []
}


cleanup_state = unity_compile_agent(
    cleanup_state
)


assert (
    cleanup_state
    .get("compile_result", {})
    .get("success")
    is True
)


assert (
    cleanup_state
    .get("compile_result", {})
    .get("system_error")
    is False
)


print(
    "PASS: 测试文件已清理，原 generated 代码真实编译通过"
)

[Unity Compiler Agent]开始执行
[Unity Compiler]编译通过
PASS: 测试文件已清理，原 generated 代码真实编译通过


## Day06-5：将 RepairAgent 的文件修改能力拆分为工程化 Repair Tool

RepairAgent 只负责分析 Root Cause、选择修复策略和调用 LLM；RepairTool 统一负责路径校验、上下文读取、幂等修改及单/多文件结果落盘。所有写入均限制在 `generated` 目录内。

In [ ]:
assert hasattr(workflow, "repair_tool")
assert repair.repair_tool is workflow.repair_tool
assert not hasattr(repair, "file_manager")

print("PASS: RepairAgent 文件修改能力已委托给 RepairTool")

## Day06-6：Diff Patch、历史记录与安全撤销

本节在临时目录演示 Git 风格 diff、源版本哈希校验、补丁历史查询和反向补丁撤销，不修改真实 Unity 生成目录。

In [9]:
import tempfile
from pathlib import Path

from memory.patch_history import PatchHistory
from tools.diff_tool import DiffTool
from tools.file_manager import FileManager

with tempfile.TemporaryDirectory() as temp_directory:
    generated_root = Path(temp_directory) / "generated"
    generated_root.mkdir()
    source_path = generated_root / "PatchDemo.cs"
    before = "public class PatchDemo {}\n"
    after = "public class PatchDemo\n{\n}\n"

    manager = FileManager()
    manager.write_file(str(source_path), before)
    diff_tool = DiffTool(manager, str(generated_root))
    history = PatchHistory(
        str(Path(temp_directory) / "patch_history.json"),
        diff_tool
    )

    patch = diff_tool.create_patch("PatchDemo.cs", before, after)
    apply_result = diff_tool.apply_patch(patch)
    record = history.record_patch(patch, before, after, apply_result)
    assert manager.read_file(str(source_path)) == after
    assert history.compare_versions(record["patch_id"]) == patch["diff"]

    undo_result = history.undo(record["patch_id"])
    assert undo_result["success"] is True
    assert manager.read_file(str(source_path)) == before

    print(patch["diff"])
    print("PASS: Day06-6 Diff Patch 创建、记录、比较与撤销通过")

[File Manager]写入完成:C:\Users\admin\AppData\Local\Temp\tmpyml2kwlq\generated\PatchDemo.cs
[File Manager]写入完成:C:\Users\admin\AppData\Local\Temp\tmpyml2kwlq\generated\PatchDemo.cs
[File Manager]写入完成:C:\Users\admin\AppData\Local\Temp\tmpyml2kwlq\generated\PatchDemo.cs
--- a/PatchDemo.cs
+++ b/PatchDemo.cs
@@ -1 +1,3 @@
-public class PatchDemo {}
+public class PatchDemo
+{
+}
PASS: Day06-6 Diff Patch 创建、记录、比较与撤销通过


## Day06-6：真实 Unity Patch 集成验收

> 运行前必须关闭 `CodingAgentTest` 的 Unity Editor。本单元会在 `day06/generated` 创建一个受控错误脚本，同步到 `CodingAgentTest/Assets/Generated`，通过 Patch 修复并真实编译，随后撤销和清理测试文件。

In [1]:
import json
import os
import tempfile
from pathlib import Path

from agents.unity_compiler import unity_compile_agent
from memory.patch_history import PatchHistory
from tools.diff_tool import DiffTool
from tools.file_manager import FileManager
from tools.repair_tool import RepairTool

day06_integration_path = Path(
    r"D:\Anaconda\Project\AI-Coding-Agent\agent-learning\day06"
)
unity_project_path = Path(
    os.getenv(
        "UNITY_TEST_PROJECT_PATH",
        r"D:\Unity\Unity_Project\CodingAgentTest"
    )
)
generated_root = day06_integration_path / "generated"
probe_name = "Day06DiffPatchProbe.cs"
source_probe_path = generated_root / probe_name
target_probe_path = (
    unity_project_path / "Assets" / "Generated" / probe_name
)
invalid_code = (
    "public class Day06DiffPatchProbe\n"
    "{\n"
    "    public int Value = ;\n"
    "}\n"
)
valid_code = (
    "public class Day06DiffPatchProbe\n"
    "{\n"
    "    public int Value = 1;\n"
    "}\n"
)

assert not source_probe_path.exists(), (
    f"测试探针已存在，请先检查:{source_probe_path}"
)

manager = FileManager()
cleanup_result = {}
target_synced = False

try:
    manager.write_file(str(source_probe_path), invalid_code)
    failed_compile_state = unity_compile_agent({"compile_history": []})
    failed_compile = failed_compile_state["compile_result"]
    assert failed_compile["success"] is False
    assert failed_compile["system_error"] is False
    assert len(failed_compile["errors"]) >= 1

    with tempfile.TemporaryDirectory() as history_directory:
        diff_tool = DiffTool(manager, str(generated_root))
        patch_history = PatchHistory(
            str(Path(history_directory) / "patch_history.json"),
            diff_tool
        )
        repair_tool = RepairTool(
            manager,
            str(generated_root),
            diff_tool,
            patch_history
        )
        patch_result = repair_tool.apply_llm_result(
            valid_code,
            probe_name
        )
        assert patch_result["success"] is True
        assert len(patch_result["patch_ids"]) == 1
        patched_source_content = manager.read_file(
            str(source_probe_path)
        )
        assert patched_source_content.strip() == valid_code.strip()

        fixed_compile_state = unity_compile_agent({"compile_history": []})
        fixed_compile = fixed_compile_state["compile_result"]
        assert fixed_compile["success"] is True
        assert fixed_compile["system_error"] is False
        target_synced = (
            target_probe_path.exists()
            and target_probe_path.read_text(encoding="utf-8")
            == patched_source_content
        )
        assert target_synced is True

        patch_id = patch_result["patch_ids"][0]
        patch_diff = patch_history.compare_versions(patch_id)
        undo_result = patch_history.undo(patch_id)
        assert undo_result["success"] is True
        assert manager.read_file(str(source_probe_path)) == invalid_code
finally:
    source_probe_path.unlink(missing_ok=True)
    cleanup_result = unity_compile_agent({"compile_history": []})[
        "compile_result"
    ]
    target_probe_path.unlink(missing_ok=True)
    target_probe_path.with_suffix(".cs.meta").unlink(missing_ok=True)

assert cleanup_result["success"] is True
assert cleanup_result["system_error"] is False
assert not source_probe_path.exists()
assert not target_probe_path.exists()

integration_summary = {
    "first_compile_success": failed_compile["success"],
    "first_error_count": len(failed_compile["errors"]),
    "patch_id": patch_id,
    "patch_applied": patch_result["success"],
    "unity_target_synced": target_synced,
    "second_compile_success": fixed_compile["success"],
    "undo_success": undo_result["success"],
    "cleanup_compile_success": cleanup_result["success"],
}

print(patch_diff)
print(json.dumps(integration_summary, ensure_ascii=False, indent=2))
print("PASS: Day06-6 真实 Unity Diff Patch 集成验收通过")

[File Manager]写入完成:D:\Anaconda\Project\AI-Coding-Agent\agent-learning\day06\generated\Day06DiffPatchProbe.cs
[Unity Compiler Agent]开始执行
[Unity Compiler]发现错误:1
[File Manager]写入完成:D:\Anaconda\Project\AI-Coding-Agent\agent-learning\day06\generated\Day06DiffPatchProbe.cs
[Unity Compiler Agent]开始执行
[Unity Compiler]编译通过
[File Manager]写入完成:D:\Anaconda\Project\AI-Coding-Agent\agent-learning\day06\generated\Day06DiffPatchProbe.cs
[Unity Compiler Agent]开始执行
[Unity Compiler]编译通过
--- a/Day06DiffPatchProbe.cs
+++ b/Day06DiffPatchProbe.cs
@@ -1,4 +1,4 @@
 public class Day06DiffPatchProbe
 {
-    public int Value = ;
+    public int Value = 1;
 }
{
  "first_compile_success": false,
  "first_error_count": 1,
  "patch_id": "9dfb48e7852f451f9d3f4e52530a8209",
  "patch_applied": true,
  "unity_target_synced": true,
  "second_compile_success": true,
  "undo_success": true,
  "cleanup_compile_success": true
}
PASS: Day06-6 真实 Unity Diff Patch 集成验收通过
